# Data Cleanup Pipeline

Comprehensive record of all modifications applied to the initial dataset (`Everything merged 18.12._new.xlsx`).

All logic lives in the `pca_mri` library. 

## Acronyms & nomenclature
* `Rec`: Réception (ou enregistrement).
* `Inv`: Investigation (ou examen).

## Sample size
N = 110 patients, 71 columns in the raw file.

## Column naming convention (post-cleanup)
| Prefix | Domain |
|--------|--------|
| `tx-` | Treatment variables |
| `psa-` | PSA / CAPRA variables |
| `mri_N-` | Serial MRI visits (N = 1–4) |
| `biopsy-` | Recurrence-investigation biopsy |
| `pet-` | PET scan |
| `bf-` | Biochemical failure |

In [1]:
import sys
sys.path.insert(0, '..')  # make pca_mri importable when running from repo root
import pandas as pd

from pca_mri import io
from pca_mri.preprocessing import columns, patients, features

## 1. Load raw dataset

In [2]:
df = pd.read_excel('Everything merged 18.12._new.xlsx')
print(df.shape)
df.head()

(110, 71)


,PatientID,DateRecInvIRM,ResultatIRMRecInv,PIRADSLesionRecInv,ResultatBiopsieRecInv,VolProstateIRM,PSA,Column8,2nd mri date,2nd mri resultat,...,all patients.age_capra,all patients.D28VolD90,all patients.D28VolV100,pet.PatientID,pet.DateRecInvRadiative,pet.TypeRecInvRadiative,pet.ResultatRecInvRadiative,date treatment.PatientId,date treatment.TypeTX,date treatment.DateBrachy
0,5753253,2024-05-18,Positive,4.0,Positive,27.7,4.10,NaN,NaT,NaN,...,1,140.9,89.3,5753253.0,2024-09-12,PET ... type,Positif,5753253,Curietherapie LDR,2022-02-24
1,5725736,2024-01-13,Négative,NaN,NaN,NaN,6.00,NaN,NaT,NaN,...,1,141.3,89.6,NaN,NaT,NaN,NaN,5725736,Curietherapie LDR,2022-01-04
2,808919,2024-02-04,Négative,NaN,NaN,30.4,5.87,NaN,NaT,NaN,...,1,164.2,95.4,NaN,NaT,NaN,NaN,808919,Curietherapie LDR,2021-05-27
3,5687019,2023-11-18,Négative,3.0,Négative,18.0,4.20,NaN,NaT,NaN,...,1,157.0,94.2,NaN,NaT,NaN,NaN,5687019,Curietherapie LDR,2021-03-25
4,5339331,2023-06-24,Positive,NaN,NaN,28.0,3.78,NaN,NaT,NaN,...,1,159.0,96.0,5339331.0,2023-08-03,PET ... type,Positif,5339331,Curietherapie LDR,2021-03-11


## 2. Remove empty columns

Drops all columns that are entirely NaN.

Known columns removed in the original dataset:  
`['Column8', 'Column12', 'Column15', 'Column18', 'Column19', '4th mri date', 'Column25', 'RecInvbiopsie.PSA', 'all patients.causedeces']`

In [3]:
print(f'Shape before: {df.shape}')
df, dropped_empty = columns.drop_empty_columns(df)
print(f'Dropped {len(dropped_empty)} empty columns: {dropped_empty}')
print(f'Shape after: {df.shape}')

Shape before: (110, 71)
Dropped 9 empty columns: ['Column8', 'Column12', 'Column15', 'Column18', 'Column19', '4th mri date', 'Column25', 'RecInvbiopsie.PSA', 'all patients.causedeces']
Shape after: (110, 62)


## 3. Remove duplicate columns

In [4]:
# same_cols,remove_cols = columns.drop_duplicate_columns(df)
# print(remove_cols)
# print(same_cols)
print(f"before: {df.shape}")
remove_cols_final = ['RecInvbiopsie.PatientID', 'all patients.PatientId',
    'pet.PatientID','date treatment.PatientId', 'date treatment.TypeTX']
df = df.drop(columns=remove_cols_final)
print(f"after: {df.shape}")

before: (110, 62)
after: (110, 57)


## 4. Rename columns

Applies the project-standard naming convention defined in  
`pca_mri.preprocessing.columns.DEFAULT_COL_MAP`.

In [5]:
print(f"before: {df.shape}")
df = columns.rename_columns(df)
print(f"after: {df.shape}")

before: (110, 57)
after: (110, 57)


## 5. Reorder columns

Groups columns logically: `patient_id` → `tx-*` → `psa-*` → `mri_N-*` → `biopsy-*` → `pet-*` → outcome dates.

In [6]:
print(f"before: {df.shape}")
df = columns.reorder_columns(df)
print(f"after: {df.shape}")

before: (110, 57)
after: (110, 57)


## 6. Flag duplicate & special patients

### A. Duplicate patients
Known duplicates: `2060194`, `2054846`, `2205490`.
* `2205490` had 2 negative biopsies (2015-08-17 00:00:00, 2016-08-25 00:00:00). Deleted row 90 (kept 91)
* `2054846` had 2 negative biopsies (2014-09-30 00:00:00, 2016-02-15 00:00:00). Deleted row 105 (kept 106).
* `2060194` had 2 negative biopsies (2014-10-21 00:00:00, 2015-07-03 00:00:00). Deleted row 107 (kept 108).

In [7]:
print(f"before: {df.shape}")
df = patients.flag_duplicate_patients(df) # adds `is_duplicate` boolean column
print(f"after: {df.shape}")
print('Duplicate patients:')
display(df[df['is_duplicate']][['patient_id', 'mri_1-result', 'biopsy-result', 'biopsy-date']])

before: (110, 57)
after: (110, 58)
Duplicate patients:


,patient_id,mri_1-result,biopsy-result,biopsy-date
90,2205490,Positive,Négatif,2015-08-17
91,2205490,Positive,Négatif,2016-08-25
105,2054846,Négative,Négatif,2014-09-30
106,2054846,Négative,Positif,2021-02-04
107,2060194,Négative,Négatif,2014-10-21
108,2060194,Négative,Positif,2015-08-14


In [8]:
# df.loc[[90, 91]].T, df.loc[[105, 106]].T, df.loc[[105, 106]].T
df.loc[91, "biopsy-date"] = pd.Timestamp("2018-10-15")
df.loc[91, "biopsy-result"] = "Positif"

df = df.drop([90, 105, 107])
print(f"after: {df.shape}")

after: (107, 58)


### B. Converter patients  
Patients whose MRI changed from positive → negative over follow-up:  
`7049838`, `7051280`, `5519879`. Strategy: include first, then run  
sensitivity analysis excluding them (see analysis notebook).

In [ ]:
df = patients.flag_converter_patients(df)
print('Converter patients (positive → negative MRI):')
display(df[df['is_converter']][['patient_id', 'mri_1-result', 'mri_2-result', 'mri_3-result', 'mri_4-result', 'biopsy-result']])

### C. Weird patients (PSA roughly doubled by MRI 2)
`2102593`, `2205490`, `2162006`.

In [ ]:
df = patients.flag_weird_patients(df)
print('Patients with PSA roughly doubled by MRI 2:')
display(df[df['psa_doubled']][['patient_id', 'mri_1-psa', 'mri_2-psa']])

## 7. Add derived features

* `biopsy-positive_ratio`: positive samples / samples taken
* `time_to_recurrence_days`: days from treatment to first surveillance MRI
* `mri_1_to_2_days`, `mri_2_to_3_days`, `mri_3_to_4_days`: inter-MRI intervals
* `psa_doubling_time_months`: PSA doubling time between MRI 1 and MRI 2

In [ ]:
df = features.add_all_features(df)
new_cols = ['biopsy-positive_ratio', 'time_to_recurrence_days',
            'mri_1_to_2_days', 'mri_2_to_3_days', 'mri_3_to_4_days',
            'psa_doubling_time_months']
df[[c for c in new_cols if c in df.columns]].describe()

## 8. Save cleaned dataset

Saves to `df-clean-02_<timestamp>.csv` and `.xlsx` in the Montreal timezone.

In [ ]:
csv_path, xlsx_path = io.save(df, stem='df-clean-02', tz='America/Montreal')
print(f'Final shape: {df.shape}')
df.head()